# TripoSG - Bild zu 3D für OpenTTS

**Einfacher Workflow:**
1. Setup ausführen (einmal)
2. Bild hochladen
3. 3D-Modell generieren
4. Herunterladen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DutchMaxwell/openTTS/blob/main/tools/3d-generation/triposg_colab.ipynb)

---
## Schritt 1: Setup (einmal ausführen)
**Dauert ca. 10-15 Minuten** beim ersten Mal (CUDA-Compiler). Danach kannst du beliebig viele Bilder konvertieren.

In [ ]:
#@title 1. Setup ausführen (klicke links auf ▶) {display-mode: "form"}

import torch
import os
import time

# GPU Check
print("=" * 50)
print("SCHRITT 1/5: GPU CHECK")
print("=" * 50)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} ({vram:.1f} GB)")
else:
    print("❌ KEINE GPU! Gehe zu: Runtime > Change runtime type > GPU")
    raise SystemExit("GPU required")

if not os.path.exists('/content/TripoSG'):
    start_time = time.time()
    
    # Step 2: Clone
    print("\n" + "=" * 50)
    print("SCHRITT 2/5: REPOSITORY KLONEN")
    print("=" * 50)
    !git clone --depth 1 https://github.com/VAST-AI-Research/TripoSG.git /content/TripoSG
    %cd /content/TripoSG
    print("✅ Repository geklont")
    
    # Step 3: Basic packages
    print("\n" + "=" * 50)
    print("SCHRITT 3/5: PAKETE INSTALLIEREN (~2 min)")
    print("=" * 50)
    !pip install -q transformers accelerate safetensors pillow trimesh einops huggingface_hub diffusers
    print("✅ Python-Pakete installiert")
    
    # Step 4: diso (marching cubes)
    print("\n" + "=" * 50)
    print("SCHRITT 4/5: DISO INSTALLIEREN (~1 min)")
    print("=" * 50)
    !pip install -q diso
    print("✅ DISO installiert")
    
    # Step 5: nvdiffrast (the slow one)
    print("\n" + "=" * 50)
    print("SCHRITT 5/5: CUDA-RENDERER KOMPILIEREN (~10 min)")
    print("=" * 50)
    print("⏳ Das dauert... Kaffee holen!")
    print("")
    !pip install git+https://github.com/NVlabs/nvdiffrast
    
    elapsed = time.time() - start_time
    print(f"\n✅ Installation abgeschlossen in {elapsed/60:.1f} Minuten!")
else:
    %cd /content/TripoSG
    print("\n✅ TripoSG bereits installiert")

# Create folders
os.makedirs('/content/input', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

print("\n" + "=" * 50)
print("✅ SETUP FERTIG - Weiter zu Schritt 2!")
print("=" * 50)

---
## Schritt 2: Bild hochladen

In [ ]:
#@title 2. Bild hochladen (klicke links auf ▶) {display-mode: "form"}

from google.colab import files
from pathlib import Path
import shutil
from IPython.display import display, Image

print("Wähle ein Bild aus (PNG oder JPG)...")
print("")

uploaded = files.upload()

if uploaded:
    for filename in uploaded.keys():
        # Move to input folder
        dest = f'/content/input/{filename}'
        shutil.move(filename, dest)
        
        print(f"\n✅ Hochgeladen: {filename}")
        
        # Show preview
        print("\nVorschau:")
        display(Image(dest, width=300))
        
        # Store for next step
        current_image = dest
        %store current_image
        
    print("\n" + "=" * 50)
    print("✅ BILD BEREIT - Weiter zu Schritt 3!")
    print("=" * 50)
else:
    print("❌ Kein Bild hochgeladen")

---
## Schritt 3: 3D-Modell generieren

In [ ]:
#@title 3. 3D-Modell generieren (klicke links auf ▶) {display-mode: "form"}

#@markdown ### Einstellungen:
qualitaet = "Mittel (10.000 Faces)" #@param ["Niedrig (5.000 Faces)", "Mittel (10.000 Faces)", "Hoch (20.000 Faces)"]

from pathlib import Path
import time

# Get face count
face_map = {
    "Niedrig (5.000 Faces)": 5000,
    "Mittel (10.000 Faces)": 10000,
    "Hoch (20.000 Faces)": 20000
}
faces = face_map[qualitaet]

# Get uploaded image
%store -r current_image

try:
    img_path = Path(current_image)
except:
    # Fallback: find any image in input
    images = list(Path('/content/input').glob('*.png')) + list(Path('/content/input').glob('*.jpg'))
    if images:
        img_path = images[-1]  # Most recent
    else:
        print("❌ Kein Bild gefunden! Führe zuerst Schritt 2 aus.")
        raise SystemExit()

output_path = f'/content/output/{img_path.stem}.glb'

print("=" * 50)
print(f"Eingabe: {img_path.name}")
print(f"Qualität: {qualitaet}")
print("=" * 50)
print("\n🔄 Generiere 3D-Modell... (ca. 2-4 Minuten)\n")

start = time.time()

!cd /content/TripoSG && python -m scripts.inference_triposg \
    --image-input "{img_path}" \
    --output-path "{output_path}" \
    --faces {faces}

elapsed = time.time() - start

if Path(output_path).exists():
    size_mb = Path(output_path).stat().st_size / 1e6
    print("\n" + "=" * 50)
    print(f"✅ FERTIG in {elapsed:.0f} Sekunden!")
    print(f"📦 Datei: {Path(output_path).name} ({size_mb:.1f} MB)")
    print("=" * 50)
    print("\nWeiter zu Schritt 4 zum Herunterladen!")
    
    # Store for download
    current_output = output_path
    %store current_output
else:
    print("\n❌ Fehler bei der Generierung")
    print("Versuche ein anderes Bild oder starte Runtime neu.")

---
## Schritt 4: Herunterladen

In [ ]:
#@title 4. GLB-Datei herunterladen (klicke links auf ▶) {display-mode: "form"}

from google.colab import files
from pathlib import Path

# Get the output file
%store -r current_output

try:
    output_path = Path(current_output)
except:
    # Fallback: find any GLB in output
    outputs = list(Path('/content/output').glob('*.glb'))
    if outputs:
        output_path = outputs[-1]
    else:
        print("❌ Kein 3D-Modell gefunden! Führe zuerst Schritt 3 aus.")
        raise SystemExit()

if output_path.exists():
    size_mb = output_path.stat().st_size / 1e6
    print(f"📦 Download: {output_path.name} ({size_mb:.1f} MB)")
    print("")
    files.download(str(output_path))
    print("\n" + "=" * 50)
    print("✅ Download gestartet!")
    print("")
    print("Nächste Schritte:")
    print("1. GLB-Datei in OpenTTS importieren")
    print("2. Spawn > Import GLB")
    print("3. Positionieren und skalieren")
    print("=" * 50)
else:
    print("❌ Datei nicht gefunden")

---
## Weitere Bilder konvertieren?

Gehe zurück zu **Schritt 2** und lade das nächste Bild hoch.

---
## Alle Downloads auf einmal

In [ ]:
#@title Alle Modelle als ZIP herunterladen {display-mode: "form"}

from google.colab import files
from pathlib import Path
import shutil

output_files = list(Path('/content/output').glob('*.glb'))

if output_files:
    print(f"Gefunden: {len(output_files)} Modelle\n")
    for f in output_files:
        size_mb = f.stat().st_size / 1e6
        print(f"  📦 {f.name} ({size_mb:.1f} MB)")
    
    # Create ZIP
    shutil.make_archive('/content/openTTS_models', 'zip', '/content/output')
    print("\n📦 Download: openTTS_models.zip")
    files.download('/content/openTTS_models.zip')
else:
    print("❌ Keine Modelle gefunden. Führe zuerst Schritt 2-3 aus.")